# T06 · Inférence statistique & tests — ADIF72

**Objectifs :**
- Tests paramétriques (z, t, binomial, chi2)
- Tests non paramétriques (Mann-Whitney, Wilcoxon, Kruskal, Shapiro)
- ANOVA (comparaison de 3+ groupes) + post-hoc Tukey
- Chi-deux (indépendance)
- Statistique bayésienne (Bêta-Binomiale)
- Théorie de la décision (coûts)

**Idée maîtresse :** chaque test est une **procédure de décision** :
« rejeter H0 ou non ? » — on contrôle la probabilité d'erreur de type I
(`α`) et on mesurera le pouvoir du test (probabilité de rejeter H0
**quand H1 est vraie**).

In [1]:
import numpy as np
from scipy import stats
np.set_printoptions(precision=4, suppress=True, linewidth=100)
print("Prêt.")

Prêt.


## 1 · Tests paramétriques

Un test **paramétrique** suppose que les données suivent une loi connue
(normale, binomiale, etc.). Avantages : plus puissant si l'hypothèse
est vraie. Inconvénients : faux négatifs si elle est fausse.

| Test | Question | Statistique |
|---|---|---|
| **z** (1 moyenne) | `μ = μ₀ ?` | z = (X̄-μ₀)/(σ/√n) |
| **t** (1 échantillon) | `μ = μ₀ ?` (σ inconnu) | t = (X̄-μ₀)/(s/√n) |
| **t** (2 échantillons) | `μ₁ = μ₂ ?` | t = (X̄₁-X̄₂)/SE |
| **binomial** | `p = p₀ ?` | P(X≥k) |
| **chi2** (conformité) | distribution = attendue ? | χ² = Σ(obs-att)²/att |
| **chi2** (indépendance) | 2 catégories indépendantes ? | χ² sur tableau |

**Le p-value est une probabilité conditionnelle :**
`p = P(statistique aussi extrême | H0 vraie)`.
Ce n'est PAS `P(H0 vraie | données)`.

In [2]:
rng = np.random.default_rng(3)
# 1a) z-test (variance connue)
n = 100
sample_z = rng.normal(0.3, 1.0, n)
z = (sample_z.mean() - 0) / (1.0/np.sqrt(n))
p_val = 2 * (1 - stats.norm.cdf(abs(z)))
print(f"1a) z-test : z={z:.3f}, p={p_val:.4f} -> {'rejeter' if p_val<0.05 else 'non'} H0")
assert p_val < 0.05

# 1b) t-test 2 échantillons
g1 = rng.normal(10, 1, 40); g2 = rng.normal(10.8, 1, 40)
t, p_t = stats.ttest_ind(g1, g2, equal_var=True)
print(f"1b) t-test : t={t:.3f}, p={p_t:.4f} -> {'différent' if p_t<0.05 else 'pas de diff'}")
assert p_t < 0.05

# 1c) chi2 indépendance
table = np.array([[20, 30, 45], [30, 25, 35]])
chi2, p_chi, dof, _ = stats.chi2_contingency(table)
print(f"1c) chi2-indépendance : chi2={chi2:.3f}, dof={dof}, p={p_chi:.4f}")

# 1d) binomial
res_b = stats.binomtest(45, 100, 0.5, alternative='two-sided')
print(f"1d) binomial : p={res_b.pvalue:.4f} -> "
      f"{'rejet' if res_b.pvalue<0.05 else 'non'} de p=0.5")

1a) z-test : z=2.382, p=0.0172 -> rejeter H0
1b) t-test : t=-3.295, p=0.0015 -> différent
1c) chi2-indépendance : chi2=3.572, dof=2, p=0.1676
1d) binomial : p=0.3682 -> non de p=0.5


### Exercice corrigé

In [3]:
# ---- Exercice P1 : test t 1 echantillon (n=16, Xbar=103, s=4, H0: mu=100) ----
n, xb, s = 16, 103, 4
t = (xb - 100) / (s / np.sqrt(n))
p = 2 * (1 - stats.t.cdf(abs(t), df=n-1))
print(f"P1-ex : t = {t:.1f}, p = {p:.4f} -> {'rejeter' if p<0.05 else 'ne pas rejeter'} H0")
assert np.isclose(t, 3.0) and p < 0.01


P1-ex : t = 3.0, p = 0.0090 -> rejeter H0


## 2 · Tests non paramétriques

Ils ne font **pas d'hypothèse sur la distribution**. Robustes aux outliers,
valables pour des échelles ordinales. **Moins puissants** que les tests
paramétriques si les hypothèses de ceux-ci sont satisfaites.

| Test paramétrique | Test non paramétrique (équivalent) |
|---|---|
| t-test 1 | **Wilcoxon signé** |
| t-test 2 échantillons | **Mann-Whitney U** |
| ANOVA | **Kruskal-Wallis** |
| (prérequis normalité) | **Shapiro-Wilk** |

**Quand les utiliser ?**
- données non normales (vérifier avec Shapiro)
- petites tailles d'échantillon
- échelles ordininales (questionnaires Likert)

In [4]:
rng = np.random.default_rng(0)
# 2a) Mann-Whitney
a = rng.normal(5, 1, 50); b = np.abs(rng.normal(5, 1, 50)) + rng.normal(0,0.1,50)
u, p_u = stats.mannwhitneyu(a, b, alternative='two-sided')
print(f"2a) Mann-Whitney : U={u:.1f}, p={p_u:.4f}")

# 2b) Wilcoxon signé
before = rng.normal(10, 1, 30); after = before + rng.normal(0.5, 0.5, 30)
w, p_w = stats.wilcoxon(before, after, alternative='two-sided')
print(f"2b) Wilcoxon signé : W={w:.1f}, p={p_w:.4f}")

# 2c) Kruskal-Wallis
gA = rng.normal(0, 1, 30); gB = rng.normal(0.5, 1, 30); gC = rng.normal(1, 1, 30)
k, p_k = stats.kruskal(gA, gB, gC)
print(f"2c) Kruskal : H={k:.3f}, p={p_k:.4f} -> {'diff' if p_k<0.05 else 'n/d'}")
assert p_k < 0.05

# 2d) Shapiro (prérequis)
norm_s = rng.normal(0, 1, 50); skew_s = rng.exponential(1, 50)
w_sp, p_sp = stats.shapiro(norm_s)
w_sk, p_sk = stats.shapiro(skew_s)
print(f"2d) Shapiro (normale) : p={p_sp:.3f} ; (expo) : p={p_sk:.3f}")

2a) Mann-Whitney : U=1331.0, p=0.5789
2b) Wilcoxon signé : W=42.0, p=0.0000
2c) Kruskal : H=12.582, p=0.0019 -> diff
2d) Shapiro (normale) : p=0.006 ; (expo) : p=0.000


### Exercice corrigé

In [5]:
# ---- Exercice P2 : Mann-Whitney (10+10, decalés) ----
a = np.array([1., 2, 3, 4, 5, 6, 7, 8, 9, 10])
b = np.array([6., 7, 8, 9, 10, 11, 12, 13, 14, 15])
u, p = stats.mannwhitneyu(a, b, alternative='two-sided')
print(f"P2-ex : U = {u}, p = {p:.5f} -> difference significative")
assert p < 0.01


P2-ex : U = 12.5, p = 0.00508 -> difference significative


## 3 · ANOVA (Analyse de variance)

Pour comparer **3 ou plus** moyennes. H0 : toutes les moyennes sont égales.
L'idée : décomposer la variance totale en
- **variabilité entre groupes** (traitement)
- **variabilité dans les groupes** (bruit)

Statistique `F = (SSB/df_between) / (SSW/df_within)`.
`F` grand → les différences entre groupes dominent le bruit → rejeter.

**Post-hoc** : si l'ANOVA est significative, il faut savoir **quelles paires**
diffèrent. On utilise **Tukey HSD** (plus strict que des t-tests multiples,
qui gonflerait `α`).

In [6]:
rng = np.random.default_rng(0)
t1 = rng.normal(20, 2, 25); t2 = rng.normal(20, 2, 25); t3 = rng.normal(24, 2, 25)
f, p_anova = stats.f_oneway(t1, t2, t3)
print(f"ANOVA : F={f:.3f}, p={p_anova:.4f} -> "
      f"{'moyennes diff' if p_anova<0.05 else 'pas de diff'}")
assert p_anova < 0.05

def tukey_pairwise(groups, alpha=0.05):
    gs = list(groups); k = len(gs)
    mse = sum(np.var(g, ddof=1) for g in gs) / k
    df = sum(len(g) - 1 for g in gs)
    q = stats.studentized_range.ppf(1 - alpha, k, df)
    hsd = q * np.sqrt(mse / min(len(g) for g in gs))
    res = {}
    for i in range(k):
        for j in range(i + 1, k):
            res[f"g{i}-g{j}"] = abs(gs[i].mean() - gs[j].mean()) > hsd
    return res, hsd

posthoc, hsd = tukey_pairwise([t1, t2, t3])
print(f"HSD = {hsd:.3f}")
for pair, sig in posthoc.items():
    print(f"  {pair}: {'SIGNIFICATIF' if sig else 'n/d'}")
assert posthoc['g0-g2'] and not posthoc['g0-g1']

ANOVA : F=36.349, p=0.0000 -> moyennes diff
HSD = 1.289
  g0-g1: n/d
  g0-g2: SIGNIFICATIF
  g1-g2: SIGNIFICATIF


### Exercice corrigé

In [7]:
# ---- Exercice P3 : ANOVA (3 groupes de meme moyenne, n=30) ----
rng_ = np.random.default_rng(0)
g1 = rng_.normal(0, 1, 30); g2 = rng_.normal(0, 1, 30); g3 = rng_.normal(0, 1, 30)
F, p = stats.f_oneway(g1, g2, g3)
print(f"P3-ex : F = {F:.3f}, p = {p:.3f} -> on ne rejette PAS H0")
assert p > 0.05


P3-ex : F = 1.374, p = 0.258 -> on ne rejette PAS H0


## 4 · Test du chi-deux (indépendance)

Pour un **tableau de contingence** : vérifie si 2 variables catégorielles
sont indépendantes.
```
H0 : les variables sont indépendantes
χ² = Σ (obs-att)²/att
df = (nb_lignes - 1)(nb_col - 1)
```
Condition d'usage : effectifs attendus `≥ 5` (sinon : Fisher exact).

In [8]:
rng = np.random.default_rng(0)
obs = np.array([[45, 30, 25], [35, 40, 25]])
chi2, p, dof, exp = stats.chi2_contingency(obs)
print(f"Chi2={chi2:.3f}, dof={dof}, p={p:.4f}")
print(f"-> {'indépendants' if p>0.05 else 'liés'}")
print("Effectifs attendus :\n", np.round(exp, 1))

Chi2=2.679, dof=2, p=0.2620
-> indépendants
Effectifs attendus :
 [[40. 35. 25.]
 [40. 35. 25.]]


### Exercice corrigé

In [9]:
# ---- Exercice P4 : chi-deux de conformite (de 6 faces, 300 lancers) ----
obs = np.array([50, 54, 48, 52, 46, 50])
att = np.full(6, 50.0)
chi2 = ((obs - att)**2 / att).sum()
p = stats.chi2.sf(chi2, df=5)
print(f"P4-ex : chi^2 = {chi2:.3f}, p = {p:.3f} -> le de semble equilibre (p > 0.05)")
assert p > 0.05


P4-ex : chi^2 = 0.800, p = 0.977 -> le de semble equilibre (p > 0.05)


## 5 · Statistique bayésienne (Bêta-Binomiale)

Le **bayésien** est différent du **frequentiste** :
- Frequentiste : le paramètre `p` est **fixé** ; on raisonne sur la
  distribution des données.
- Bayésien : le paramètre `p` est **aléatoire** (a une distribution
  de probabilité) ; on met à jour le **prior** avec les données.

Pour la proportion `p` avec des essais binomiaux :
- **Prior** `p ~ Bêta(α, β)` (uniforme si α=β=1)
- **Vraisemblance** : `k` succès sur `n` essais
- **Posterior** : `p | données ~ Bêta(α+k, β+n-k)` (conjugaison)

In [10]:
alpha, beta = 1, 1
s, e = 60, 40
a_post, b_post = alpha + s, beta + e
p_gt_05 = 1 - stats.beta.cdf(0.5, a_post, b_post)
print(f"Prior Bêta({alpha},{beta}) + {s}/{s+e} -> Posterior Bêta({a_post},{b_post})")
print(f"E[p] = {a_post/(a_post+b_post):.4f}")
print(f"P(p > 0.5) = {p_gt_05:.5f}")
assert p_gt_05 > 0.97

Prior Bêta(1,1) + 60/100 -> Posterior Bêta(61,41)
E[p] = 0.5980
P(p > 0.5) = 0.97698


### Exercice corrigé

In [11]:
# ---- Exercice P5 : Beta-Binomiale (8/10, prior Beta(1,1)) ----
ap, bp = 1+8, 1+2
E_post = ap/(ap+bp)
P_gt = 1 - stats.beta.cdf(0.5, ap, bp)
print(f"P5-ex : posterior Beta({ap},{bp}), E[p] = {E_post:.3f}, P(p>0.5) = {P_gt:.5f}")
assert E_post > 0.7 and P_gt > 0.95


P5-ex : posterior Beta(9,3), E[p] = 0.750, P(p>0.5) = 0.96729


## 6 · Théorie de la décision (coûts)

Quand les erreurs sont **asymétriques** (faux négatif coûte plus que
faux positif), le critère « p < 0.05 » ne suffit plus. On utilise
la **matrice de coût** :
```
             Décision : Malade    Décision : Sain
Réel Malade :  0 (bon)      100 (FN)
Réel Sain   :  20 (FP)      0 (bon)
```
La décision optimale minimise le **coût attendu** :
`E[cost | décision] = P(réel) × cost`.
C'est le fondement de la décision en médecine, finance, ML (seuils).

In [12]:
C = np.array([[0, 100], [20, 0]])
p_malade = 0.5
cost_dec_malade = p_malade * C[0, 0] + (1 - p_malade) * C[1, 0]
cost_dec_sain = p_malade * C[0, 1] + (1 - p_malade) * C[1, 1]
print(f"Coût moyen 'malade' = {cost_dec_malade:.1f}")
print(f"Coût moyen 'sain'   = {cost_dec_sain:.1f}")
print(f"-> Choisir {'malade' if cost_dec_malade<cost_dec_sain else 'sain'} "
      f"(minimise le coût)")
assert cost_dec_malade < cost_dec_sain

Coût moyen 'malade' = 10.0
Coût moyen 'sain'   = 50.0
-> Choisir malade (minimise le coût)


### Exercice corrigé

In [13]:
# ---- Exercice P6 : theorie de la decision (matrice de couts) ----
C = np.array([[0, 100], [20, 0]])
p_mal = 0.5
cm = p_mal*C[0,0] + (1-p_mal)*C[1,0]
cs = p_mal*C[0,1] + (1-p_mal)*C[1,1]
print(f"P6-ex : cout 'malade' = {cm:.1f}, cout 'sain' = {cs:.1f} -> decider 'malade'")
assert cm < cs


P6-ex : cout 'malade' = 10.0, cout 'sain' = 50.0 -> decider 'malade'


## 7 · Synthèse globale — l'essentiel & exemple intégrateur

### Ce qu'il faut retenir (les 6 points)

| Point | L'essentiel |
|---|---|
| **1 · Tests paramétriques** | z / t : exigent des hypothèses sur la distribution (normalité, n) |
| **2 · Tests non paramétriques** | Mann-Whitney, Wilcoxon : sur les rangs, robustes aux outliers |
| **3 · ANOVA** | ≥ 3 groupes : F = variance inter-groupes / intra-groupes |
| **4 · Chi-deux** | tableaux de contingence : indépendance de deux catégories |
| **5 · Bêta-Binomiale** | conjugué naturel d'une proportion : posterior direct, sans test |
| **6 · Théorie de la décision** | p-value ≠ décision : coûts des erreurs type I / II tranchent |

**Réflexe examen :** une p-value se *lit* (« H0 improbable à ce point ? »), elle ne
se *transforme* pas en vérité. L'exemple ci-dessous analyse le même A/B test par
fréquentiste, bayésien et décision.

### Exercice corrigé

In [14]:
# ---- Exercice P7 : p-value de z = 1.8 (deux queues) ----
z = 1.8
p = 2 * (1 - stats.norm.cdf(z))
print(f"P7-ex : P(|Z| >= 1.8) = {p:.4f} -> a alpha = 0.05 : ne pas rejeter H0")
assert p > 0.05


P7-ex : P(|Z| >= 1.8) = 0.0719 -> a alpha = 0.05 : ne pas rejeter H0


### Exemple intégrateur (tous les points)

In [15]:
# ---- Exemple intégrateur : un A/B test, trois lectures (P1, P5, P6) ----
rng7 = np.random.default_rng(706)
nA7, nB7 = 800, 800
convA7 = int(rng7.binomial(nA7, 0.10))
convB7 = int(rng7.binomial(nB7, 0.14))
pA7, pB7 = convA7/nA7, convB7/nB7

p_pool = (convA7 + convB7) / (nA7 + nB7)                   # P1 : test z des proportions
z7 = (pB7 - pA7) / np.sqrt(p_pool*(1-p_pool)*(1/nA7 + 1/nB7))
pval7 = 2*(1 - stats.norm.cdf(abs(z7)))
print(f"{convA7}/{nA7} vs {convB7}/{nB7} : z = {z7:.2f}, p-value = {pval7:.4f}")
assert pval7 < 0.05                                         # vraie différence -> rejet H0

sb = np.random.default_rng(106)                             # P5 : bayésien Bêta(1,1)
sA_7 = sb.beta(convA7 + 1, nA7 - convA7 + 1, 50000)
sB_7 = sb.beta(convB7 + 1, nB7 - convB7 + 1, 50000)
p_b_sup = float((sB_7 > sA_7).mean())
print(f"P(B > A | données) = {p_b_sup:.3f}")
assert p_b_sup > 0.95

gain7 = (pB7 - pA7) * 100_000                               # P6 : décision sous coût nul
print(f"Gain attendu si on publie B : {gain7:,.0f} conversions/100k -> publier")
assert gain7 > 0
print("\n✅ T06 terminé — 6 points + synthèse globale.")

74/800 vs 121/800 : z = 3.59, p-value = 0.0003
P(B > A | données) = 1.000
Gain attendu si on publie B : 5,875 conversions/100k -> publier

✅ T06 terminé — 6 points + synthèse globale.
